# Choosing error and observable decoders

qLDPC uses typed helper functions to configure syndrome-to-error decoders. Observable decoders wrap those settings and compile against a Stim detector error model.

In [ ]:
%pip install -q qldpc

In [ ]:
import numpy as np
import stim

from qldpc import codes, decoders

## Error decoding

A decoder spec stores algorithm options until the parity-check matrix is available. The same spec can be passed to code methods, `decoders.decode`, or a Sinter-facing observable decoder.

In [ ]:
code = codes.RepetitionCode(5)
syndrome = np.array([1, 0, 0, 0])

error_decoder = decoders.bp_lsd(max_iter=30, bp_method="ms")
inferred_error = decoders.decode(code.matrix, syndrome, decoder=error_decoder)
inferred_error

## Observable decoding

`ObservableDecoder` compiles an inner error decoder for a detector error model, then maps inferred errors to observable flips.

In [ ]:
dem = stim.DetectorErrorModel("error(0.1) D0 L0")
observable_decoder = decoders.ObservableDecoder(decoder=decoders.lookup_table(max_weight=1))
compiled = observable_decoder.compile_decoder_for_dem(dem)
compiled.decode_observables(np.array([1], dtype=np.uint8))

Use `ObservableLookupDecoder` directly when a lookup table should return observable flips rather than a representative physical error.

In [ ]:
lookup = decoders.ObservableLookupDecoder(dem, max_weight=1)
lookup.decode_observables(np.array([1], dtype=int))